# Project C - Dataset Exploration

This section explores the dataset for the three target classes:
- Chicken
- Egg
- Balloon

It computes:
- Class distribution and imbalance ratios
- Basic image-quality diagnostics (size, aspect ratio, potential corruption)
- Qualitative class samples

The setup is designed to run on both **Google Colab** and **Kaggle** by changing only the dataset root path.

In [1]:
#!pip install -U pip setuptools wheel

In [2]:
!pip install numpy pandas matplotlib seaborn


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Imports
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)

print("Imports loaded.")

Imports loaded.


In [4]:
## Download the dataset from the following link:
# 

In [5]:
# Platform-aware dataset path setup (Kaggle + Colab)
TARGET_CLASSES = ["Chicken", "Egg", "Balloon"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Candidate roots for each environment.
# Priority is top-to-bottom unless DATASET_ROOT is manually set.
DEFAULT_CANDIDATE_ROOTS = [
    Path("/kaggle/input"),                  # Kaggle datasets
    Path("/content/drive/MyDrive"),         # Colab + mounted Drive
    Path("/content"),                       # Colab local
    Path("./data"),                         # Local fallback
]

# Optional: set this manually to your exact dataset root if needed.
# Example: DATASET_ROOT = Path('/kaggle/input/open-images-subset')
# Example: DATASET_ROOT = Path('/content/drive/MyDrive/cvai/open-images-subset')
# Local: DATASET_ROOT = Path(r"PATH_TO_DATASET")
# PATH_TO_DATASET example = "c:\mai\2.Semester\CVAI\project\data"
DATASET_ROOT = Path(r"c:\mai\2.Semester\CVAI\project\data")


def pick_root(explicit_root=None, candidates=None):
    if explicit_root is not None:
        p = Path(explicit_root)
        if p.exists():
            return p
        raise FileNotFoundError(f"Provided DATASET_ROOT does not exist: {p}")

    candidates = candidates or DEFAULT_CANDIDATE_ROOTS
    for candidate in candidates:
        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate a dataset root. Set DATASET_ROOT manually to your dataset folder."
    )


ROOT = pick_root(DATASET_ROOT)
print(f"Using ROOT: {ROOT}")

Using ROOT: c:\mai\2.Semester\CVAI\project\data


## Download Open Images subset via Python (Chicken, Egg, Balloon)

This section downloads train images for the target classes using the Open Images class IDs you provided:
- Egg: `/m/033cnk`
- Chicken: `/m/09b5t`
- Balloon: `/m/01j51`

The downloader uses `fiftyone`'s Open Images integration and stores images in class-specific folders so they are easy to use for classification.

In [6]:
%pip install -q fiftyone tqdm
!pip install -U eta

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# ================================
# CLEAN OPEN IMAGES → CLASSIFICATION DATASET
# ================================

from pathlib import Path
from collections import defaultdict
import shutil
import fiftyone as fo
import fiftyone.zoo as foz

# ----------------
# CONFIG
# ----------------
CLASS_NAMES = ["Egg (Food)", "Chicken", "Balloon"]
MAX_IMAGES_PER_CLASS = 200
SPLIT = "train"

OUTPUT_DIR = Path(".../project/data/openimages_subset/classification")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# optional: verhindere Cache-Probleme
DATASET_NAME = "open-images-clean-v1"

if DATASET_NAME in fo.list_datasets():
    fo.delete_dataset(DATASET_NAME)

# ----------------
# 1. LOAD DATASET
# ----------------
dataset = foz.load_zoo_dataset(
    "open-images-v7",
    split=SPLIT,
    label_types=["detections"],
    classes=CLASS_NAMES,
    max_samples=2000,  
    shuffle=True,
    dataset_name=DATASET_NAME,
)

print("Loaded samples:", len(dataset))

# ----------------
# 2. FILTER LABELS
# ----------------
view = dataset.filter_labels(
    "ground_truth",
    fo.ViewField("label").is_in(CLASS_NAMES)
)

# nur Bilder behalten, die überhaupt unsere Klassen enthalten
view = view.match(
    fo.ViewField("ground_truth.detections.label").contains(CLASS_NAMES)
)

print("After filtering:", len(view))

# ----------------
# 3. BALANCE DATASET
# ----------------
per_class_samples = defaultdict(list)

for sample in view:
    labels = set(det.label for det in sample.ground_truth.detections)

    for label in labels:
        if label in CLASS_NAMES and len(per_class_samples[label]) < MAX_IMAGES_PER_CLASS:
            per_class_samples[label].append(sample)

# ----------------
# 4. EXPORT
# ----------------
for label, samples in per_class_samples.items():
    class_dir = OUTPUT_DIR / label.replace(" ", "_")
    class_dir.mkdir(parents=True, exist_ok=True)

    for i, sample in enumerate(samples):
        src = Path(sample.filepath)
        dst = class_dir / f"{i}_{src.name}"
        shutil.copy(src, dst)

# ----------------
# 5. SUMMARY
# ----------------
print("\nFinal dataset:")
for label, samples in per_class_samples.items():
    print(f"{label}: {len(samples)} images")

print("\nSaved to:", OUTPUT_DIR.resolve())

Found 500 images, downloading the remaining 1500
 100% |█████████████████| 1500/1500 [38.0s elapsed, 0s remaining, 16.2 files/s]      
Dataset info written to 'C:\Users\KristofZörenyi\fiftyone\open-images-v7\info.json'
Loading 'open-images-v7' split 'train'
 100% |███████████████| 2000/2000 [14.4s elapsed, 0s remaining, 129.5 samples/s]      
Dataset 'open-images-clean-v1' created
Loaded samples: 2000
After filtering: 2000

Final dataset:
Chicken: 200 images
Balloon: 200 images
Egg (Food): 200 images

Saved to: C:\mai\2.Semester\CVAI\project\src\data\openimages_subset\classification


## Observation Template (fill after running)

Use this section for the report text required by the assignment.

- **Classes found:** Chicken, Egg, Balloon (confirm exact folder matches).
- **Distribution:** include counts and shares from `dist_df`.
- **Imbalance:** report `imbalance_ratio` and whether rebalancing is needed.
- **Quality concerns:** mention unreadable files, very small images, extreme aspect ratios, and potential label noise/background leakage observed in samples.
- **Expected training impact:** explain likely effects on convergence and generalization (e.g., underrepresented class bias, noisy labels, low-resolution limits).

In [16]:
# Helpers to discover class folders and collect image paths

def find_class_dir(root: Path, class_name: str):
    """Find directory matching class name case-insensitively under root."""
    if not root.exists():
        return None

    # First, direct child match
    for p in root.iterdir():
        if p.is_dir() and p.name.lower() == class_name.lower():
            return p

    # Then recursive match
    for p in root.rglob("*"):
        if p.is_dir() and p.name.lower() == class_name.lower():
            return p

    return None


def gather_image_paths(class_dir: Path):
    paths = []
    if class_dir is None:
        return paths
    for p in class_dir.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS:
            paths.append(p)
    return paths


records = []
for class_name in TARGET_CLASSES:
    class_dir = find_class_dir(ROOT, class_name)
    class_paths = gather_image_paths(class_dir)

    records.append({
        "class": class_name,
        "class_dir": str(class_dir) if class_dir else None,
        "num_images": len(class_paths),
        "paths": class_paths,
    })

summary_df = pd.DataFrame(records)
summary_df[["class", "class_dir", "num_images"]]

,class,class_dir,num_images
0,Chicken,None,0
1,Egg,None,0
2,Balloon,None,0


In [ ]:
# Class distribution and imbalance metrics
counts = summary_df.set_index("class")["num_images"].astype(int)

if counts.sum() == 0:
    raise ValueError(
        "No images found for target classes. Check ROOT and class folder names."
    )

ax = counts.sort_values(ascending=False).plot(kind="bar", color=["#4c78a8", "#f58518", "#54a24b"])
ax.set_title("Image Count per Class")
ax.set_ylabel("Number of images")
ax.set_xlabel("Class")
plt.tight_layout()
plt.show()

max_count = counts.max()
min_count = counts.min()
imbalance_ratio = (max_count / min_count) if min_count > 0 else np.inf

dist_df = pd.DataFrame({
    "class": counts.index,
    "count": counts.values,
    "share": (counts.values / counts.sum())
}).sort_values("count", ascending=False)

print("Total images:", int(counts.sum()))
print(f"Imbalance ratio (max/min): {imbalance_ratio:.2f}")
display(dist_df.style.format({"share": "{:.2%}"}))

In [ ]:
# Qualitative samples per class
SAMPLES_PER_CLASS = 6

fig, axes = plt.subplots(len(TARGET_CLASSES), SAMPLES_PER_CLASS, figsize=(2.6 * SAMPLES_PER_CLASS, 2.6 * len(TARGET_CLASSES)))

if len(TARGET_CLASSES) == 1:
    axes = np.array([axes])

for row_idx, class_name in enumerate(TARGET_CLASSES):
    row = summary_df[summary_df["class"] == class_name].iloc[0]
    image_paths = row["paths"]

    selected = random.sample(image_paths, k=min(SAMPLES_PER_CLASS, len(image_paths))) if image_paths else []

    for col_idx in range(SAMPLES_PER_CLASS):
        ax = axes[row_idx, col_idx]
        ax.axis("off")

        if col_idx < len(selected):
            p = selected[col_idx]
            try:
                img = Image.open(p).convert("RGB")
                ax.imshow(img)
                ax.set_title(class_name, fontsize=10)
            except Exception:
                ax.text(0.5, 0.5, "Unreadable", ha="center", va="center")
        else:
            ax.text(0.5, 0.5, "N/A", ha="center", va="center", fontsize=9)

plt.suptitle("Random Samples by Class", y=1.02)
plt.tight_layout()
plt.show()